# Pipeline —  automated lyric video Generator


**Goal:** upload one MP4 and run one cell to produce `lyric_video.mp4`, `timing_metadata.json`, `style_plan.json`, and `quality_report.json`.



In [ ]:
!apt-get -qq update
!apt-get -qq install -y fonts-noto-core fonts-noto-extra fontconfig

!fc-cache -f -v
!fc-list :lang=te family | head -20

In [ ]:
# ONE CELL: uploads one MP4, then runs the entire pipeline automatically.
import os, sys, subprocess, json, re, base64, io
from pathlib import Path

def cmd_run(args):
    p = subprocess.run(args, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    if p.returncode:
        raise RuntimeError("Command failed:\n" + p.stderr[-4000:])
    return p

print("1/8 Installing dependencies…")
subprocess.run(["bash","-lc","apt-get -qq update && apt-get -qq install -y ffmpeg"], check=True)
subprocess.run([sys.executable,"-m","pip","install","-q","openai-whisper","opencv-python-headless","numpy","pillow","requests"], check=True)

from google.colab import files, userdata
from IPython.display import display, Video
import numpy as np, cv2, whisper, requests
from PIL import Image

print("2/8 Upload one MP4…")
uploaded = files.upload()
if not uploaded: raise RuntimeError("No file uploaded.")
video_name = next(iter(uploaded))
video_path = str(Path("/content") / video_name)
duration = float(cmd_run(["ffprobe","-v","error","-show_entries","format=duration","-of","default=noprint_wrappers=1:nokey=1",video_path]).stdout.strip())
if duration <= 0: raise RuntimeError("Could not read video duration.")
work = Path("/content/lyric_video_output"); work.mkdir(exist_ok=True)
audio_path = str(work/"audio.wav")
output_path = "/content/lyric_video.mp4"
ass_path = str(work/"lyrics.ass")

print("3/8 Extracting audio and transcribing…")
cmd_run(["ffmpeg","-y","-i",video_path,"-vn","-ac","1","-ar","16000",audio_path])
print("Loading Whisper large-v3 for Telugu transcription...")

model = whisper.load_model("large-v3")

transcription = model.transcribe(
    audio_path,
    task="transcribe",
    word_timestamps=True,
    condition_on_previous_text=False,
    temperature=0.0,
    verbose=False
)

transcript_text = transcription.get("text", "").strip()

print("Detected language:", transcription.get("language"))
print("Transcript:", transcript_text)

if not transcript_text:
    print("WARNING: No lyrics were recognized.")
transcription = model.transcribe(audio_path, word_timestamps=True, verbose=False, condition_on_previous_text=False)
words = []
for seg in transcription.get("segments",[]):
    for w in seg.get("words",[]) or []:
        token = str(w.get("word","")).strip()
        if not token: continue
        try: start, end = float(w["start"]), float(w["end"])
        except Exception: continue
        start=max(0,start); end=max(start+0.04,end)
        if start < duration:
            words.append({"word":token,"start":round(start,3),"end":round(min(end,duration),3),"probability":round(float(w.get("probability",0)),3)})
if not words: raise RuntimeError("No word timestamps found. Try a clip with clearer vocals.")
print("Words:",len(words),"Language:",transcription.get("language"))

print("4/8 Sampling frames and estimating colors…")
cap=cv2.VideoCapture(video_path); fps=cap.get(cv2.CAP_PROP_FPS) or 25
sample_every=max(1,int(fps*max(2,duration/10)))
frames=[]; colors=[]; motion=[]; prev=None; idx=0
while len(frames)<10:
    ok,frame=cap.read()
    if not ok: break
    if idx%sample_every==0:
        small=cv2.resize(frame,(160,90)); rgb=cv2.cvtColor(small,cv2.COLOR_BGR2RGB); frames.append(rgb)
        pix=rgb.reshape(-1,3).astype(np.float32)
        rng=np.random.default_rng(idx+9); pix=pix[rng.choice(len(pix),min(1000,len(pix)),replace=False)]
        _,lab,cent=cv2.kmeans(pix,3,None,(cv2.TERM_CRITERIA_EPS+cv2.TERM_CRITERIA_MAX_ITER,15,1),1,cv2.KMEANS_PP_CENTERS)
        colors.append([int(x) for x in cent[int(np.argmax(np.bincount(lab.flatten()))) ]])
        gray=cv2.cvtColor(small,cv2.COLOR_BGR2GRAY)
        if prev is not None: motion.append(float(cv2.absdiff(gray,prev).mean()))
        prev=gray
    idx+=1
cap.release()
if not frames: raise RuntimeError("Could not decode video frames.")
luma=float(np.mean([0.299*r+0.587*g+0.114*b for r,g,b in colors]))
text_hex="#FFFFFF" if luma<150 else "#141414"
outline_hex="#0A0A0A" if text_hex=="#FFFFFF" else "#FFFFFF"
accent=max(colors,key=lambda c:(max(c)-min(c))/max(1,max(c)))
def hx(c): return "#%02X%02X%02X"%tuple(c)

style={
 "source":"opencv_fallback","mood":"estimated from sampled frames",
 "palette":[hx(c) for c in colors[:5]],"text_color":text_hex,
 "outline_color":outline_hex,"accent_color":hx(accent),
 "typography":"bold sans-serif","placement":"lower third",
 "animation":"word highlighting with fade","motion_level":"energetic" if (np.mean(motion) if motion else 0)>12 else "subtle"
}

print("5/8 Requesting Replicate Vision AI style plan (fallback available)…")
try: token=userdata.get("REPLICATE_API_TOKEN")
except Exception: token=None
api_used=False
if token:
    try:
        image=Image.fromarray(frames[len(frames)//2]); buf=io.BytesIO(); image.save(buf,format="JPEG",quality=80)
        data_uri="data:image/jpeg;base64,"+base64.b64encode(buf.getvalue()).decode()
        prompt=("Analyze this music-video frame for a lyric overlay. Return JSON only with keys mood, palette (3 hex colors), typography, placement (center, upper third, lower third), animation (subtle, medium, energetic), rationale. Prioritize readable contrast and avoid covering the main subject.")
        r=requests.post("https://api.replicate.com/v1/models/yorickvp/llava-13b/predictions",
          headers={"Authorization":"Bearer "+token,"Content-Type":"application/json","Prefer":"wait"},
          json={"input":{"image":data_uri,"prompt":prompt,"max_tokens":350}},timeout=120)
        if r.ok:
            out=r.json().get("output",""); raw="".join(map(str,out)) if isinstance(out,list) else str(out)
            m=re.search(r"\{.*\}",raw,re.S)
            if m:
                cand=json.loads(m.group())
                if isinstance(cand,dict):
                    style.update(cand); style["source"]="replicate:yorickvp/llava-13b"; api_used=True
        else: print("Replicate returned HTTP",r.status_code,"— fallback style will be used.")
    except Exception as e: print("Replicate unavailable — fallback style will be used:",str(e)[:180])
else: print("No REPLICATE_API_TOKEN in Colab Secrets; using local style fallback.")
# Validate renderer-facing choices
if str(style.get("placement","")).lower() not in ["center","upper third","lower third"]: style["placement"]="lower third"
if str(style.get("animation","")).lower() not in ["subtle","medium","energetic"]: style["animation"]="subtle"
style["text_color"]=text_hex
style["outline_color"]=outline_hex
style["accent_color"]=style.get("accent_color",hx(accent))
with open(work/"style_plan.json","w") as f: json.dump(style,f,indent=2)

print("6/8 Building timed animated subtitles…")
def asscol(color):
    s=str(color).lstrip("#")
    if not re.fullmatch(r"[0-9a-fA-F]{6}",s): s="FFFFFF"
    return "&H00"+s[4:6]+s[2:4]+s[0:2]+"&"
placement=str(style["placement"]).lower()
align=5 if placement=="center" else (8 if placement=="upper third" else 2)
margin=0 if align==5 else (130 if align==8 else 100)
groups=[]; cur=[]
for w in words:
    if cur and (len(cur)>=6 or w["start"]-cur[-1]["end"]>0.7 or w["end"]-cur[0]["start"]>3.4):
        groups.append(cur); cur=[]
    cur.append(w)
    if re.search(r"[.!?,;:]$",w["word"]): groups.append(cur); cur=[]
if cur: groups.append(cur)
def ts(t):
    t=max(0,float(t)); h=int(t//3600); m=int(t%3600//60); s=int(t%60); cs=int((t-int(t))*100)
    return f"{h}:{m:02d}:{s:02d}.{cs:02d}"
header=f"""[Script Info]
ScriptType: v4.00+
PlayResX: 1920
PlayResY: 1080
WrapStyle: 2
ScaledBorderAndShadow: yes
[V4+ Styles]
Format: Name, Fontname, Fontsize, PrimaryColour, SecondaryColour, OutlineColour, BackColour, Bold, Italic, Underline, StrikeOut, ScaleX, ScaleY, Spacing, Angle, BorderStyle, Outline, Shadow, Alignment, MarginL, MarginR, MarginV, Encoding
Style: Lyric,Arial,64,{asscol(text_hex)},{asscol(style['accent_color'])},{asscol(outline_hex)},&H66000000&,-1,0,0,0,100,100,1,0,1,4,1,{align},100,100,{margin},1
[Events]
Format: Layer, Start, End, Style, Name, MarginL, MarginR, MarginV, Effect, Text
"""
events=[]
for g in groups:
    start=g[0]["start"]; end=min(duration,max(g[-1]["end"]+0.12,start+0.25)); pieces=[]
    for w in g:
        clean=w["word"].replace("{","").replace("}","")
        pieces.append(r"{\k"+str(max(1,int((w["end"]-w["start"])*100)))+"}"+clean)
    fade=100 if str(style["animation"]).lower()=="energetic" else 170
    txt=r"{\fad("+str(fade)+","+str(fade)+r")\bord4\shad1}"+" ".join(pieces)
    events.append(f"Dialogue: 0,{ts(start)},{ts(end)},Lyric,,0,0,0,,{txt}")
with open(ass_path,"w",encoding="utf-8") as f: f.write(header+"\n".join(events)+"\n")

print("7/8 Rendering output MP4…")
p=subprocess.run(["ffmpeg","-y","-i",video_path,"-vf",f"subtitles={ass_path}","-c:v","libx264","-preset","veryfast","-crf","21","-c:a","aac","-b:a","192k","-movflags","+faststart",output_path],stdout=subprocess.PIPE,stderr=subprocess.PIPE,text=True)
if p.returncode: raise RuntimeError("FFmpeg render failed:\n"+p.stderr[-4000:])
out_dur=float(cmd_run(["ffprobe","-v","error","-show_entries","format=duration","-of","default=noprint_wrappers=1:nokey=1",output_path]).stdout.strip())

print("8/8 Writing metadata and quality report…")
mean_conf=float(np.mean([w["probability"] for w in words])) if words else 0
anomalies=[{"index":i,"word":w["word"],"reason":"duration_over_2_5_seconds"} for i,w in enumerate(words) if w["end"]-w["start"]>2.5]
metadata={"input_file":video_name,"duration_seconds":duration,"detected_language":transcription.get("language"),"word_count":len(words),"transcript":" ".join(w["word"] for w in words),"words":words,"mean_whisper_word_probability":round(mean_conf,3),"timing_anomalies":anomalies}
quality={"render_success":os.path.getsize(output_path)>10000,"output_duration_seconds":round(out_dur,3),"duration_difference_seconds":round(abs(duration-out_dur),3),"word_count":len(words),"mean_whisper_word_probability":round(mean_conf,3),"timing_anomaly_count":len(anomalies),"vision_api_used":api_used,"style_source":style.get("source"),"checks":{"output_exists":os.path.exists(output_path),"output_nonempty":os.path.getsize(output_path)>10000,"duration_close":abs(duration-out_dur)<=max(2,duration*.03),"transcript_nonempty":bool(words)},"limitations":["Whisper timestamps may be inaccurate for singing.","Heuristic checks do not prove exact synchronization.","Vision model sees sampled frames only.","Fallback style is based on local visual features."]}
for name,obj in [("timing_metadata.json",metadata),("quality_report.json",quality)]:
    with open(work/name,"w",encoding="utf-8") as f: json.dump(obj,f,indent=2,ensure_ascii=False)

print("\nCOMPLETE — preview below")
print("Style plan:",work/"style_plan.json")
print("Timing metadata:",work/"timing_metadata.json")
print("Quality report:",work/"quality_report.json")
display(Video(output_path,embed=True,width=800))
for path in [output_path,str(work/"style_plan.json"),str(work/"timing_metadata.json"),str(work/"quality_report.json")]:
    files.download(path)
